# Intermediate Matplotlib

**Estimated time:** 45–60 minutes  
**Prerequisites:** Basic Python, some exposure to `plt.plot()`

## Learning Goals

| # | Topic |
|---|-------|
| 1 | Figure/Axes architecture — `fig, ax = plt.subplots()` |
| 2 | Subplots — grids, shared axes, `GridSpec` |
| 3 | Customization — spines, ticks, labels, annotations |
| 4 | Chart types — histogram, scatter, bar, heatmap |
| 5 | Twin axes and secondary y-axis |
| 6 | Styles, colormaps, and color cycles |
| 7 | Saving figures — DPI, format, tight layout |
| 8 | Putting it together — dashboard-style figure |

---

### Architecture Quick Reference

```
Figure  — the whole window/canvas
  └─ Axes — one plot panel (has its own coordinate system)
       ├─ Axis (x/y) — individual axis with ticks, labels
       ├─ Artists — lines, patches, text, etc.
       └─ Legend
```

```python
fig, ax = plt.subplots()              # 1 panel
fig, axes = plt.subplots(2, 3)        # 2x3 grid → axes.shape = (2,3)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

print('matplotlib', plt.matplotlib.__version__)
%matplotlib inline

# Reproducibility
rng = np.random.default_rng(42)

In [ ]:
# Shared dataset: synthetic insurance portfolio
np.random.seed(42)
lines = ['Auto', 'Property', 'Liability']
years = list(range(2017, 2024))

rows = []
for ln in lines:
    for yr in years:
        rows.append({
            'line': ln,
            'year': yr,
            'premium': np.random.randint(5_000_000, 20_000_000),
            'loss_ratio': np.clip(np.random.normal(0.62, 0.10), 0.30, 1.20),
        })
df = pd.DataFrame(rows)
df['claims'] = (df['premium'] * df['loss_ratio']).astype(int)
df.head()

---
## Section 1 — Figure / Axes Architecture

Always use the **object-oriented API** (`fig, ax = plt.subplots()`), not `plt.plot()` directly. It's explicit, composable, and works correctly with subplots.

| OO API | pyplot shortcut | Notes |
|--------|-----------------|-------|
| `ax.set_title(s)` | `plt.title(s)` | OO is always unambiguous |
| `ax.set_xlabel(s)` | `plt.xlabel(s)` | |
| `ax.set_xlim(a, b)` | `plt.xlim(a, b)` | |
| `ax.legend()` | `plt.legend()` | |
| `ax.tick_params(...)` | — | Fine-grained tick control |

In [ ]:
# Basic line plot — OO style
auto = df[df['line'] == 'Auto'].sort_values('year')

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(auto['year'], auto['loss_ratio'],
        marker='o', linewidth=2, markersize=7,
        color='steelblue', label='Auto')

# Reference line at 70%
ax.axhline(0.70, color='firebrick', linestyle='--', linewidth=1.2, label='Target 70%')

ax.set_title('Auto Loss Ratio by Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Accident Year')
ax.set_ylabel('Loss Ratio')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
ax.set_xticks(auto['year'])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# EXERCISE 1:
# Plot all 3 lines of business on the same Axes, each a different color.
# Add a horizontal reference line at 65% and format the y-axis as percentages.
# YOUR CODE HERE

---
## Section 2 — Subplots: grids and GridSpec

```python
fig, axes = plt.subplots(nrows, ncols,
                         sharex=True, sharey=True,
                         figsize=(width, height))
```

`GridSpec` gives you uneven grid layouts (wide panel + narrow sidebar, etc.).

**Tip:** `axes.flat` lets you iterate over a 2D axes array with one loop.

In [ ]:
# 1×3 grid — one panel per line of business
colors = {'Auto': 'steelblue', 'Property': 'darkorange', 'Liability': 'seagreen'}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, ln in zip(axes, lines):
    sub = df[df['line'] == ln].sort_values('year')
    ax.bar(sub['year'], sub['loss_ratio'], color=colors[ln], alpha=0.8, width=0.6)
    ax.axhline(0.70, color='firebrick', linestyle='--', linewidth=1, label='70%')
    ax.set_title(ln, fontweight='bold')
    ax.set_xticks(sub['year'])
    ax.set_xticklabels(sub['year'], rotation=45)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
    ax.grid(axis='y', alpha=0.3)

axes[0].set_ylabel('Loss Ratio')
fig.suptitle('Loss Ratio by Line of Business', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# GridSpec: wide chart on top, 3 small charts on bottom
fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Top row: spans all 3 columns
ax_top = fig.add_subplot(gs[0, :])
pivot = df.pivot_table('premium', index='year', columns='line', aggfunc='sum')
pivot.plot(ax=ax_top, marker='o', linewidth=2)
ax_top.set_title('Total Premium by Line ($)', fontweight='bold')
ax_top.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
ax_top.grid(alpha=0.3)

# Bottom row: one bar chart per line
for col_idx, ln in enumerate(lines):
    ax_b = fig.add_subplot(gs[1, col_idx])
    sub = df[df['line'] == ln].sort_values('year')
    ax_b.bar(sub['year'], sub['claims'] / 1e6, color=colors[ln], alpha=0.8, width=0.6)
    ax_b.set_title(f'{ln} Claims', fontweight='bold', fontsize=10)
    ax_b.set_ylabel('Claims ($M)')
    ax_b.tick_params(axis='x', rotation=45)

plt.show()

In [ ]:
# EXERCISE 2:
# Create a 2x2 subplot grid where each panel shows a scatter plot of
# premium (x) vs claims (y) for a specific year (2017, 2019, 2021, 2023).
# Use the same x/y axis limits across all panels (sharex, sharey).
# YOUR CODE HERE

---
## Section 3 — Customization: Spines, Ticks, and Annotations

```python
ax.spines['top'].set_visible(False)       # remove top border
ax.tick_params(axis='x', labelsize=10, rotation=45)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}'))
ax.annotate('text', xy=(x,y), xytext=(xt,yt), arrowprops=dict(...))
```

In [ ]:
# Publication-quality styling: remove chart junk
auto = df[df['line'] == 'Auto'].sort_values('year')

fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(auto['year'], auto['loss_ratio'],
        marker='o', linewidth=2.5, markersize=8, color='steelblue', zorder=3)

# Fill between target and actual
ax.fill_between(auto['year'], 0.70, auto['loss_ratio'],
                where=auto['loss_ratio'] > 0.70,
                alpha=0.15, color='firebrick', label='Above target')

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Lighter left/bottom spines
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')

ax.tick_params(colors='#555555', length=4)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
ax.set_ylim(0.40, 1.0)
ax.set_xticks(auto['year'])
ax.grid(axis='y', color='#eeeeee', linewidth=1)

# Annotate worst year
worst_idx = auto['loss_ratio'].idxmax()
worst = auto.loc[worst_idx]
ax.annotate(
    f"Worst year\n{worst['loss_ratio']:.0%}",
    xy=(worst['year'], worst['loss_ratio']),
    xytext=(worst['year'] - 1.2, worst['loss_ratio'] + 0.08),
    arrowprops=dict(arrowstyle='->', color='firebrick'),
    color='firebrick', fontsize=9,
)

ax.set_title('Auto Loss Ratio — Accident Year', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Accident Year', color='#555555')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# EXERCISE 3:
# Take the multi-line loss ratio chart from Exercise 1 and:
# a) Remove top and right spines
# b) Annotate the single (line, year) point with the HIGHEST loss ratio with its value
# c) Add a shaded fill_between for the range min..max across all lines
# YOUR CODE HERE

---
## Section 4 — Chart Types: Histogram, Scatter, Bar, Heatmap

| Chart | When to use |
|-------|-------------|
| `ax.hist` | Distribution of a single variable |
| `ax.scatter` | Relationship between two numeric variables |
| `ax.bar` / `ax.barh` | Categorical comparisons |
| `ax.imshow` | Matrix / heatmap data |
| `ax.stackplot` | Part-to-whole over time |

In [ ]:
# Histogram — overlapping distributions
claims_auto     = rng.lognormal(np.log(15_000), 1.1, 3000)
claims_property = rng.lognormal(np.log(40_000), 1.3, 2000)

fig, ax = plt.subplots(figsize=(9, 4))

bins = np.logspace(2, 7, 50)  # log-spaced bins for skewed data

ax.hist(claims_auto,     bins=bins, alpha=0.6, color='steelblue',  label='Auto',     density=True)
ax.hist(claims_property, bins=bins, alpha=0.6, color='darkorange', label='Property', density=True)

ax.set_xscale('log')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xlabel('Claim Severity')
ax.set_ylabel('Density')
ax.set_title('Claim Severity Distribution (log scale)', fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot — premium vs claims, sized by year, colored by line
color_map = {'Auto': 'steelblue', 'Property': 'darkorange', 'Liability': 'seagreen'}

fig, ax = plt.subplots(figsize=(8, 5))

for ln in lines:
    sub = df[df['line'] == ln]
    sc = ax.scatter(
        sub['premium'] / 1e6,
        sub['loss_ratio'],
        c=color_map[ln],
        s=(sub['year'] - 2015) * 15,  # size encodes year
        alpha=0.75, edgecolors='white', linewidths=0.5,
        label=ln
    )

ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
ax.set_xlabel('Written Premium')
ax.set_ylabel('Loss Ratio')
ax.set_title('Premium vs Loss Ratio  (bubble size ∝ year)', fontweight='bold')
ax.legend(title='Line')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap — loss ratio by (line, year)
pivot = df.pivot_table('loss_ratio', index='line', columns='year')

fig, ax = plt.subplots(figsize=(9, 3))

im = ax.imshow(pivot.values, cmap='RdYlGn_r', vmin=0.40, vmax=0.90, aspect='auto')

# Labels
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

# Annotate each cell
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        ax.text(j, i, f'{val:.0%}', ha='center', va='center',
                fontsize=9, color='black' if 0.50 < val < 0.80 else 'white')

plt.colorbar(im, ax=ax, label='Loss Ratio', format=mticker.PercentFormatter(xmax=1.0, decimals=0))
ax.set_title('Loss Ratio Heatmap by Line and Year', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# EXERCISE 4:
# Create a horizontal stacked bar chart showing the premium split by line for each year.
# Each bar = one year; segments = Auto / Property / Liability
# Hint: iterate over lines and use ax.barh with 'left' parameter to stack
# YOUR CODE HERE

---
## Section 5 — Twin Axes and Secondary Y-Axis

Use `ax.twinx()` to add a second y-axis that shares the same x-axis. Useful for overlaying series with very different scales (e.g. volume and rate).

**Watch out:** dual axes can be misleading. Use them only when both series are genuinely related and labeled clearly.

In [ ]:
# Show written premium (bars) and loss ratio (line) on same chart
all_yr = df.groupby('year').agg(total_premium=('premium','sum'), avg_lr=('loss_ratio','mean')).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 5))

# Left axis: premium bars
ax1.bar(all_yr['year'], all_yr['total_premium'] / 1e6,
        color='steelblue', alpha=0.6, width=0.6, label='Premium ($M)')
ax1.set_ylabel('Written Premium ($M)', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))

# Right axis: loss ratio line
ax2 = ax1.twinx()
ax2.plot(all_yr['year'], all_yr['avg_lr'],
         color='firebrick', marker='o', linewidth=2.5, label='Loss Ratio')
ax2.set_ylabel('Average Loss Ratio', color='firebrick')
ax2.tick_params(axis='y', labelcolor='firebrick')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
ax2.set_ylim(0, 1.2)

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax1.set_title('Premium Volume vs Loss Ratio', fontweight='bold')
ax1.set_xticks(all_yr['year'])
ax1.spines[['top']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# EXERCISE 5:
# Create a twin-axis chart for the Auto line only:
#   Left axis: claims amount (bar)
#   Right axis: number of policy years (simulate as premium / 5000, i.e. policy count)
# YOUR CODE HERE

---
## Section 6 — Styles, Colormaps, and Color Cycles

```python
plt.style.use('seaborn-v0_8-whitegrid')  # built-in style
plt.rcParams.update({'font.family': 'monospace', 'axes.titlesize': 14})

# Get N evenly spaced colors from a colormap
cmap = plt.get_cmap('tab10')
colors = [cmap(i / N) for i in range(N)]
```

**Perceptually uniform colormaps** (for quantitative data): `viridis`, `plasma`, `cividis`  
**Diverging** (positive/negative or above/below threshold): `RdYlGn`, `coolwarm`, `bwr`  
**Qualitative** (categories): `tab10`, `Set2`, `Paired`

In [ ]:
# Preview available styles
styles = [s for s in plt.style.available if 'seaborn' in s or 'ggplot' in s or 'bmh' in s]
print('Interesting styles:', styles[:10])

In [ ]:
# Compare two styles side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = np.linspace(0, 2 * np.pi, 100)

for ax, style in zip(axes, ['default', 'bmh']):
    with plt.style.context(style):
        for i, (label, shift) in enumerate([('Sin', 0), ('Cos', np.pi/2), ('Sin2x', 0)]):
            y = np.sin(x + shift + i * 0.3)
            ax.plot(x, y, label=label)
        ax.set_title(f"style: '{style}'", fontweight='bold')
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Custom diverging colormap for loss ratio (green = good, red = bad)
lr_cmap = LinearSegmentedColormap.from_list(
    'lr_custom',
    [(0.0, '#2ecc71'),   # green at 0 = very good LR
     (0.5, '#f9ca24'),   # yellow at midpoint
     (1.0, '#c0392b')],  # red at 1 = very bad LR
)

fig, ax = plt.subplots(figsize=(9, 3))
pivot = df.pivot_table('loss_ratio', index='line', columns='year')
im = ax.imshow(pivot.values, cmap=lr_cmap, vmin=0.40, vmax=0.90, aspect='auto')

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i,j]:.0%}", ha='center', va='center', fontsize=9)

plt.colorbar(im, ax=ax, label='Loss Ratio',
             format=mticker.PercentFormatter(xmax=1.0, decimals=0))
ax.set_title('Custom Colormap — Loss Ratio Heatmap', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# EXERCISE 6:
# Plot the aggregate loss distribution from the Monte Carlo simulation below
# using a histogram with 'plasma' colormap applied to the bars
# Shade the region beyond the 95th percentile in a distinct color
rng2 = np.random.default_rng(99)
agg_losses = rng2.lognormal(np.log(2_000_000), 0.5, 10_000)
# YOUR CODE HERE

---
## Section 7 — Saving Figures

```python
fig.savefig('output.png', dpi=150, bbox_inches='tight')
fig.savefig('output.pdf')          # vector — best for print
fig.savefig('output.svg')          # vector — best for web
```

| Parameter | Effect |
|-----------|--------|
| `dpi` | Dots per inch — 72 screen, 150 presentation, 300 print |
| `bbox_inches='tight'` | Removes excess whitespace |
| `transparent=True` | Transparent background (PNG/SVG) |
| `facecolor='white'` | Explicit background for dark-mode environments |

**`plt.tight_layout()`** should be called before `savefig` — it adjusts spacing to prevent label overlap.

In [ ]:
import os

fig, ax = plt.subplots(figsize=(8, 4))
all_yr_auto = df[df['line'] == 'Auto'].sort_values('year')
ax.plot(all_yr_auto['year'], all_yr_auto['loss_ratio'],
        marker='o', linewidth=2, color='steelblue')
ax.set_title('Auto Loss Ratio')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
plt.tight_layout()

# Save to the notebooks directory
out_path = 'auto_loss_ratio.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
print(f'Saved: {out_path}  ({os.path.getsize(out_path) / 1024:.1f} KB)')
plt.show()

---
## Section 8 — Capstone: Dashboard-Style Figure

Combine everything: `GridSpec`, twin axes, heatmap, custom formatting, and annotation — all in one publication-ready figure.

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#f8f9fa')

gs = gridspec.GridSpec(
    2, 3, figure=fig,
    hspace=0.45, wspace=0.35,
    left=0.07, right=0.96, top=0.88, bottom=0.10
)

# ── Title ──────────────────────────────────────────────────────────────
fig.text(0.5, 0.94, 'Insurance Portfolio Dashboard',
         ha='center', fontsize=16, fontweight='bold', color='#2c3e50')
fig.text(0.5, 0.91, 'Synthetic Data · Accident Years 2017–2023',
         ha='center', fontsize=10, color='#7f8c8d')

# ── Panel A (top-left, spans 2 cols): Premium trend by line ────────────
ax_a = fig.add_subplot(gs[0, 0:2])
pivot_prem = df.pivot_table('premium', index='year', columns='line', aggfunc='sum') / 1e6
ax_a.stackplot(
    pivot_prem.index,
    [pivot_prem[ln] for ln in lines],
    labels=lines, alpha=0.85,
    colors=['steelblue', 'darkorange', 'seagreen'],
)
ax_a.set_title('A. Stacked Written Premium ($M)', fontweight='bold', fontsize=11)
ax_a.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
ax_a.legend(loc='upper left', fontsize=8, framealpha=0.5)
ax_a.spines[['top', 'right']].set_visible(False)
ax_a.set_facecolor('#f8f9fa')

# ── Panel B (top-right): Loss ratio heatmap ───────────────────────────
ax_b = fig.add_subplot(gs[0, 2])
pivot_lr = df.pivot_table('loss_ratio', index='line', columns='year')
im = ax_b.imshow(pivot_lr.values, cmap='RdYlGn_r', vmin=0.4, vmax=0.9, aspect='auto')
ax_b.set_xticks(range(len(pivot_lr.columns)))
ax_b.set_xticklabels(pivot_lr.columns, rotation=45, fontsize=7)
ax_b.set_yticks(range(len(pivot_lr.index)))
ax_b.set_yticklabels(pivot_lr.index, fontsize=8)
for i in range(len(pivot_lr.index)):
    for j in range(len(pivot_lr.columns)):
        ax_b.text(j, i, f"{pivot_lr.values[i,j]:.0%}",
                  ha='center', va='center', fontsize=7,
                  color='white' if (pivot_lr.values[i,j] < 0.48 or pivot_lr.values[i,j] > 0.82) else 'black')
ax_b.set_title('B. Loss Ratio Heatmap', fontweight='bold', fontsize=11)
plt.colorbar(im, ax=ax_b, shrink=0.9,
             format=mticker.PercentFormatter(xmax=1.0, decimals=0))

# ── Panels C-E (bottom): one scatter per line ─────────────────────────
for col_idx, ln in enumerate(lines):
    ax_c = fig.add_subplot(gs[1, col_idx])
    sub = df[df['line'] == ln]
    sc = ax_c.scatter(
        sub['premium'] / 1e6, sub['loss_ratio'],
        c=sub['year'], cmap='viridis', s=80, alpha=0.85,
        edgecolors='white', linewidths=0.5
    )
    ax_c.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
    ax_c.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
    ax_c.set_title(f'C{col_idx+1}. {ln}: Premium vs LR', fontweight='bold', fontsize=10)
    ax_c.set_xlabel('Premium')
    if col_idx == 0:
        ax_c.set_ylabel('Loss Ratio')
    ax_c.spines[['top', 'right']].set_visible(False)
    ax_c.set_facecolor('#f8f9fa')
    plt.colorbar(sc, ax=ax_c, shrink=0.85, label='Year')

plt.show()

In [ ]:
# EXERCISE 8 (capstone):
# Build your own 2-panel figure:
#   Left panel: stacked bar chart of claims by line per year
#   Right panel: scatter of avg premium (x) vs avg loss ratio (y) per year,
#                with the year label as text next to each point
# Requirements:
#   - Clean spines (remove top/right)
#   - Y-axis formatted as % for loss ratio panel
#   - Proper titles and axis labels
#   - Save to 'my_dashboard.png' at 150 DPI
# YOUR CODE HERE

---
## Wrap-Up Cheat Sheet

| Topic | Key takeaway |
|-------|--------------|
| Architecture | Always use `fig, ax = plt.subplots()` — not bare `plt.plot()` |
| Subplots | `plt.subplots(nrows, ncols)` → iterate with `zip(axes, data)` |
| GridSpec | Uneven layouts — wide headers + small panels |
| Spines | `ax.spines[['top','right']].set_visible(False)` for clean look |
| Formatters | `mticker.PercentFormatter`, `FuncFormatter` for currency/units |
| Annotation | `ax.annotate(text, xy=point, xytext=label_pos, arrowprops=...)` |
| Twin axes | `ax.twinx()` for dual y-axes; combine legends manually |
| Colormaps | Perceptual uniform = viridis; diverging = RdYlGn; categorical = tab10 |
| Saving | `fig.savefig(path, dpi=150, bbox_inches='tight', facecolor='white')` |